In [1]:
!pip install -q langchain langchain-community sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 

In [3]:
import os
from google.colab import drive
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

drive.mount('/content/drive')
PROJECT_PATH = "/content/drive/MyDrive/VictorianGPT"
clean_folder = f"{PROJECT_PATH}/cleaned"
db_folder = f"{PROJECT_PATH}/chroma_db"

os.makedirs(db_folder, exist_ok=True)
print("Environment ready.")

/tmp/ipykernel_958/4273787163.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


Mounted at /content/drive
Environment ready.


In [4]:
# Read the cleaned books
texts = []
sources = []

for file in os.listdir(clean_folder):
    if file.endswith(".txt"):
        with open(f"{clean_folder}/{file}", 'r', encoding="utf8") as f:
            texts.append(f.read())
            sources.append(file)

print(f"Loaded {len(texts)} books.")

# Split the texts into 1000-character chunks with a 100-character overlap for context
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = []
chunk_sources = []

for i, text in enumerate(texts):
    split_docs = text_splitter.split_text(text)
    chunks.extend(split_docs)
    chunk_sources.extend([sources[i]] * len(split_docs))

print(f"Created {len(chunks)} searchable chunks from the literature.")

Loaded 4 books.
Created 3731 searchable chunks from the literature.


In [5]:
# We use a fast, highly efficient embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Generating embeddings and building Chroma database... (This may take a few minutes)")

# Create the vector store
vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    metadatas=[{"source": src} for src in chunk_sources],
    persist_directory=db_folder
)

print(f"Vector database successfully saved to: {db_folder}")

/tmp/ipykernel_958/1077285764.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.wa

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings and building Chroma database... (This may take a few minutes)
Vector database successfully saved to: /content/drive/MyDrive/VictorianGPT/chroma_db


In [6]:
# Test query
query = "What happens to the ship Demeter in the storm?"

# Retrieve the top 2 most relevant chunks
docs = vectorstore.similarity_search(query, k=2)

print(f"QUERY: {query}\n")
for i, doc in enumerate(docs):
    print(f"--- RETRIEVED CHUNK {i+1} (Source: {doc.metadata['source']}) ---")
    print(doc.page_content)
    print("\n")

QUERY: What happens to the ship Demeter in the storm?

--- RETRIEVED CHUNK 1 (Source: dracula.txt) ---
. A. and R. I. walls in May next. More than one captain made up his mind then and there that his “cobble” or his “mule,” as they term the different classes of boats, would remain in the harbour till the storm had passed. The wind fell away entirely during the evening, and at midnight there was a dead calm, a sultry heat, and that prevailing intensity which, on the approach of thunder, affects persons of a sensitive nature. There were but few lights in sight at sea, for even the coasting steamers, which usually “hug” the shore so closely, kept well to seaward, and but few fishing-boats were in sight. The only sail noticeable was a foreign schooner with all sails set, which was seemingly going westwards. The foolhardiness or ignorance of her officers was a prolific theme for comment whilst she remained in sight, and efforts were made to signal her to reduce sail in face of her danger


